# MJX Policy Training for Mujoco Cobot

Training a robot manipulation policy using MJX (MuJoCo + JAX) with large batch sizes for efficient parallel simulation and learning.

## Section 1: Import Required Libraries

Import MJX, JAX, NumPy, and other dependencies for batched simulation and policy training.

In [1]:
import os
os.environ['JAX_PLATFORM_NAME'] = 'gpu'  # Use GPU if available, falls back to CPU

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, value_and_grad
import numpy as np
import optax
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import time
from functools import partial

print(f"JAX devices: {jax.devices()}")
print(f"JAX version: {jax.__version__}")

# Import mujoco and mjx
try:
    import mujoco
    import mujoco.mjx as mjx
    print(f"MuJoCo version: {mujoco.__version__}")
except ImportError:
    print("Installing mujoco...")
    os.system("pip install mujoco -q")
    import mujoco
    import mujoco.mjx as mjx

JAX devices: [CudaDevice(id=0)]
JAX version: 0.6.2
Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
MuJoCo version: 3.6.0


## Section 2: Load the Robot Environment

Load the mujoco_cobot robot model from the XML file and configure the environment.

In [ ]:
# Path to the robot model
xml_path = "/home/luisamao/villa_spaces/sim_ws/robot_simple.xml"

import os
import mujoco
import xml.etree.ElementTree as ET

# Load and process XML
print(f"Loading model from: {xml_path}")

tree = ET.parse(xml_path)
root = tree.getroot()
# base_dir = os.path.dirname(xml_path)
base_dir = "/home/luisamao/villa_spaces/sim_ws/src/mujoco_cobot/assets"

# 1. Resolve relative include paths
for include_elem in root.findall('include'):
    if 'file' in include_elem.attrib:
        rel_path = include_elem.attrib['file']
        abs_path = os.path.normpath(os.path.join(base_dir, rel_path))
        include_elem.attrib['file'] = abs_path

# 2. FIX: Ensure 'gym_floor_mat' exists
# Find or create the <asset> element
asset_elem = root.find('asset')
if asset_elem is None:
    asset_elem = ET.SubElement(root, 'asset')

# Resolve all mesh and texture paths to absolute paths
for tag in ['mesh', 'texture']:
    for elem in root.findall(f".//{tag}"):
        if 'file' in elem.attrib:
            rel_path = elem.attrib['file']
            # Only change if it's not already an absolute path
            if not os.path.isabs(rel_path):
                abs_path = os.path.normpath(os.path.join(base_dir, rel_path))
                elem.attrib['file'] = abs_path
                # Optional: print for debugging
                # print(f"Resolved {tag}: {rel_path} -> {abs_path}")

# Check if the material is already there; if not, add it
if root.find(".//material[@name='gym_floor_mat']") is None:
    print("Injecting missing 'gym_floor_mat' into assets...")
    # Add a default texture and material so the compiler doesn't crash
    ET.SubElement(asset_elem, 'texture', 
                  type='2d', name='gym_floor_tex', 
                  builtin='checker', rgb1='.2 .3 .4', rgb2='.1 .2 .3', 
                  width='512', height='512')
    ET.SubElement(asset_elem, 'material', 
                  name='gym_floor_mat', 
                  texture='gym_floor_tex', 
                  texrepeat='2 2', specular='0.3', shininess='0.5')

# 3. Convert to string
xml_string = ET.tostring(root, encoding='unicode')

# 4. Compile MuJoCo model
# try:
#     model = mujoco.MjModel.from_xml_string(xml_string)
#     print(f"\nModel loaded successfully!")
#     print(f"  - DOFs: {model.nv}")
#     print(f"  - Actuators: {model.nu}")
# except ValueError as e:
#     print(f"\nCompilation Error: {e}")

# Compile MuJoCo model
print(xml_string)
model = mujoco.MjModel.from_xml_string(xml_string)
model.opt.iterations = 1  # Or 2-4 for more accuracy, but 1 is fastest
model.opt.ls_iterations = 0 # Disable line search (uses while_loop)
model.opt.solver = mujoco.mjtSolver.mjSOL_NEWTON
print(f"\nModel loaded successfully!")
print(f"  - DOFs: {model.nv}")
print(f"  - Bodies: {model.nbody}")
print(f"  - Joints: {model.njnt}")
print(f"  - Actuators: {model.nu}")

# Environment configuration
TIMESTEP = 0.002  # 2ms per step
BATCH_SIZE = 4 # 256  # Large batch size for parallel simulations
EPISODE_LENGTH = 20 # 200  # Steps per episode
NUM_EPISODES = 10 # 100  # Total training episodes
LEARNING_RATE = 1e-3

print(f"\nEnvironment Configuration:")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Episode Length: {EPISODE_LENGTH}")
print(f"  - Total Training Episodes: {NUM_EPISODES}")


Loading model from: /home/luisamao/villa_spaces/sim_ws/robot_collision_prim.xml
Injecting missing 'gym_floor_mat' into assets...
<mujoco model="Omni_Kinova_Cobot">
  <compiler angle="radian" autolimits="true" />
  <option integrator="implicitfast" cone="elliptic" impratio="10" timestep="0.002" />
  <visual>
    <global offwidth="1920" offheight="1080" />
  </visual>
  <default>
    <default class="visual">
      <geom type="mesh" contype="0" conaffinity="0" group="2" rgba="0.75294 0.75294 0.75294 1" />
    </default>
    <default class="collision">
      <geom type="capsule" group="3" />
    </default>
    <default class="large_actuator">
      <position kp="2000" kv="100" forcerange="-105 105" />
    </default>
    <default class="small_actuator">
      <position kp="500" kv="50" forcerange="-52 52" />
    </default>
    <site size="0.001" rgba="0.5 0.5 0.5 0.3" group="4" />
    <default class="2f85">
      <mesh scale="0.001 0.001 0.001" />
      <general biastype="affine" />
      <

ValueError: Error: transmission target 'split' not found in actuator 10
Element name 'fingers_actuator', id 10, line 234

## Section 3: Initialize MJX Model and Data

Set up the MJX model for efficient batch processing.

In [ ]:
# Convert to MJX model for batch processing
mjx_model = mjx.put_model(model)

# Initialize MJX data
mjx_data = mjx.put_data(model, mujoco.MjData(model))

print(f"MJX Model initialized:")
print(f"  - nv (DOFs): {mjx_model.nv}")
print(f"  - nu (actuators): {mjx_model.nu}")
print(f"  - Joint names: {[model.joint(i).name for i in range(min(7, model.njnt))]}")

# Get actuator info
print(f"\nActuators: {mjx_model.nu}")
if mjx_model.nu > 0:
    print(f"  - Ctrl range: {mjx_model.actuator_ctrlrange}")

# Get observation dimension (joint positions + velocities)
obs_dim = mjx_model.nq + mjx_model.nv
action_dim = mjx_model.nu
print(f"\nObservation dimension: {obs_dim}")
print(f"Action dimension: {action_dim}")

MJX Model initialized:
  - nv (DOFs): 18
  - nu (actuators): 11
  - Joint names: ['base_x', 'base_y', 'base_yaw', 'joint_1', 'joint_2', 'joint_3', 'joint_4']

Actuators: 11
  - Ctrl range: [[-10.         10.       ]
 [-10.         10.       ]
 [ -3.14        3.14     ]
 [  0.          0.       ]
 [ -2.2497294   2.2497294]
 [  0.          0.       ]
 [ -2.5795965   2.5795965]
 [  0.          0.       ]
 [ -2.099631    2.099631 ]
 [  0.          0.       ]
 [  0.        255.       ]]

Observation dimension: 36
Action dimension: 11


## Section 4: Create Batch Processing Pipeline

Implement vectorized environment step function using JAX vmap for large batch processing.

In [ ]:
# 1. Pre-calculate the body ID once (Global or passed as constant)
EE_NAME = 'bracelet_link' # maybe base
try:
    # Use the CPU model to get the ID as a standard integer
    EE_BODY_ID = model.body(EE_NAME).id
    print(f"End-effector ID fixed: {EE_BODY_ID}")
except:
    EE_BODY_ID = 0
    print([model.body(i).name for i in range(model.nbody)])
    print("Warning: Link not found, defaulting to 0")

# 2. Pre-allocate the MJX data structure once
# We will use this as a template so JAX doesn't re-allocate memory
base_data = mjx.make_data(mjx_model)

def reset_fn(key):
    """Fast reset: uses pre-allocated structure template."""
    # Note: No print statements here! They slow down the trace.
    
    # Random initial joint configuration (example: small noise around 0)
    q = jax.random.uniform(key, (mjx_model.nq,), minval=-0.01, maxval=0.01)
    qvel = jnp.zeros(mjx_model.nv)
    
    # 3. Update the template instead of creating new data
    data = base_data.replace(qpos=q, qvel=qvel)
    
    # Compute forward kinematics
    data = mjx.forward(mjx_model, data)
    return data

def get_ee_pos(data):
    """Fast extraction using integer indexing."""
    return data.xpos[EE_BODY_ID]

def get_observation(data):
    """Extract observation from mjx data."""
    # Observation: [joint_pos, joint_vel, ee_pos, ee_vel]
    obs = jnp.concatenate([data.qpos, data.qvel])
    return obs

# def get_ee_pos(data):
#     """Get end-effector position."""
#     # Body ID for end-effector
#     ee_body_id = model.body('robotiq_85_base_link').id if 'robotiq_85_base_link' in [model.body(i).name for i in range(model.nbody)] else 0
#     if ee_body_id > 0:
#         return data.xpos[ee_body_id]
#     return data.xpos[0]

def reward_fn(data, target_pos):
    """Compute reward: distance to target."""
    ee_pos = get_ee_pos(data)
    
    # Distance reward
    distance = jnp.linalg.norm(ee_pos - target_pos)
    reward = -distance
    
    return reward

@jit
def step_fn(data, action, target_pos):
    """Single environment step."""
    # Clip action to valid range
    action = jnp.clip(action, -1.0, 1.0)
    
    # Set control
    data = data.replace(ctrl=action[:mjx_model.nu])
    
    # Step physics
    data = mjx.step(mjx_model, data)
    
    # Get observation and reward
    obs = get_observation(data)
    reward = reward_fn(data, target_pos)
    
    return data, obs, reward

# Vectorized versions for batch processing
batch_reset = vmap(reset_fn)
batch_step = vmap(step_fn, in_axes=(0, 0, None))  # Batch data & actions, single target
batch_get_obs = vmap(get_observation)

print("Batch processing functions compiled with vmap!")
print(f"  - batch_reset: resets {BATCH_SIZE} environments")
print(f"  - batch_step: steps {BATCH_SIZE} environments in parallel")
print(f"  - batch_get_obs: extracts observations from {BATCH_SIZE} environments")

End-effector ID fixed: 11
Batch processing functions compiled with vmap!
  - batch_reset: resets 4 environments
  - batch_step: steps 4 environments in parallel
  - batch_get_obs: extracts observations from 4 environments


## Section 5: Implement Neural Network Policy and Training Loop

Define a simple MLP policy network and implement the training loop with large batch processing.

In [ ]:
import flax.linen as nn
from flax.core import FrozenDict

class PolicyNetwork(nn.Module):
    """Simple MLP policy network."""
    action_dim: int
    hidden_size: int = 128
    
    def setup(self):
        self.fc1 = nn.Dense(self.hidden_size)
        self.fc2 = nn.Dense(self.hidden_size)
        self.fc_out = nn.Dense(self.action_dim)
    
    def __call__(self, obs):
        x = nn.relu(self.fc1(obs))
        x = nn.relu(self.fc2(x))
        action = jnp.tanh(self.fc_out(x))  # Tanh for normalized actions [-1, 1]
        return action

# Initialize policy network
policy = PolicyNetwork(action_dim=action_dim)
key = jax.random.PRNGKey(0)
dummy_obs = jnp.zeros(obs_dim)
params = policy.init(key, dummy_obs)

print(f"Policy network initialized with parameters:")
print(f"  - Input: {obs_dim} (observations)")
print(f"  - Hidden: 128")
print(f"  - Output: {action_dim} (actions)")

# Vectorized policy usage
# Fix: Map over observations (axis 0), but keep params static (None)
batch_policy = vmap(lambda obs, p: policy.apply(p, obs), in_axes=(0, None))

# Optimizer setup
optimizer = optax.adam(learning_rate=LEARNING_RATE)
opt_state = optimizer.init(params)

print(f"\nOptimizer: Adam with learning rate {LEARNING_RATE}")

Policy network initialized with parameters:
  - Input: 36 (observations)
  - Hidden: 128
  - Output: 11 (actions)

Optimizer: Adam with learning rate 0.001


In [ ]:
# 2. OPTIMIZED LOSS FUNCTION (The "lax.scan" fix)
def loss_fn(params, batch_data_init, batch_target_pos, key):
    """Trajectory rollout using lax.scan to save memory and compilation time."""
    
    def policy_step(carry, _):
        data, obs = carry
        # Get actions: batch_policy must have in_axes=(0, None)
        actions = batch_policy(obs, params)
        # Step physics
        next_data, next_obs, rewards = batch_step(data, actions, batch_target_pos)
        return (next_data, next_obs), rewards

    # Initial state
    init_obs = batch_get_obs(batch_data_init)
    
    # 
    # lax.scan compiles the STEP once and loops it, instead of unrolling 200 steps.
    _, rewards = jax.lax.scan(
        policy_step, 
        (batch_data_init, init_obs), 
        None, 
        length=EPISODE_LENGTH
    )
    
    # rewards shape is (EPISODE_LENGTH, BATCH_SIZE)
    return -jnp.mean(rewards)

# JIT compile loss function
loss_fn_jit = jit(loss_fn)

print("Training functions compiled with JIT!")

Training functions compiled with JIT!


In [ ]:
print(f"\n{'='*60}")
print(f"Starting Training: {NUM_EPISODES} episodes, batch_size={BATCH_SIZE}")
print(f"{'='*60}\n")

# Training loop
training_losses = []
training_rewards = []
start_time = time.time()

for episode in range(NUM_EPISODES):
    # Random target position
    target_pos = jnp.array([0.3, 0.2, 0.3])  # Target for reaching task
    print("before reset")
    # Initialize batch of environments
    keys = jax.random.split(key, BATCH_SIZE)
    batch_data_init = batch_reset(keys)
    print("computing loss")
    
    # Compute loss
    loss, grads = value_and_grad(loss_fn_jit)(params, batch_data_init, target_pos, key)
    
    print("before update")
    # Update parameters
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    print("after update")
    # Track metrics
    training_losses.append(float(loss))
    training_rewards.append(-float(loss) * EPISODE_LENGTH)
    
    # Print progress
    if (episode + 1) % 10 == 0 or True:
        elapsed = time.time() - start_time
        avg_reward = np.mean(training_rewards[-10:])
        print(f"Episode {episode+1:3d}/{NUM_EPISODES}  |  Loss: {loss:8.4f}  |  Avg Reward: {avg_reward:7.4f}  |  Time: {elapsed:.1f}s")

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"Training Complete! Total time: {total_time:.1f}s")
print(f"{'='*60}\n")

print(f"Final training reward: {training_rewards[-1]:.4f}")
print(f"Best reward: {max(training_rewards):.4f}")
print(f"Training efficiency: {NUM_EPISODES * BATCH_SIZE / total_time:.0f} steps/sec")


Starting Training: 10 episodes, batch_size=4

before reset
computing loss


2026-03-12 15:16:11.514964: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


before update
after update
Episode   1/10  |  Loss:      nan  |  Avg Reward:     nan  |  Time: 493.9s
before reset
computing loss


In [ ]:
# Force compilation to happen NOW so we can see where it hangs
print("Starting initial compilation (this may take 2-5 minutes on a GTX 1070)...")

# Dummy data for a test run
test_keys = jax.random.split(key, BATCH_SIZE)
test_data = batch_reset(test_keys)
test_target = jnp.array([0.3, 0.2, 0.3])

# Trigger JIT
_ = loss_fn_jit(params, test_data, test_target, key)
print("Compilation complete! Starting training loop...") # ye it took 5 minutes on GTX 1070, 6 ish minutes on A6000 gpu

# for episode in range(NUM_EPISODES):
#     # Random target position
#     target_pos = jnp.array([0.3, 0.2, 0.3])  # Target for reaching task
#     print("before reset")
#     # Initialize batch of environments
#     keys = jax.random.split(key, BATCH_SIZE)
#     batch_data_init = batch_reset(keys)
#     print("computing loss")
    
#     # Compute loss
#     loss, grads = value_and_grad(loss_fn_jit)(params, batch_data_init, target_pos, key)
    
#     print("before update")
#     # Update parameters
#     updates, opt_state = optimizer.update(grads, opt_state, params)
#     params = optax.apply_updates(params, updates)
#     print("after update")
#     # Track metrics
#     training_losses.append(float(loss))
#     training_rewards.append(-float(loss) * EPISODE_LENGTH)
    
#     # Print progress
#     if (episode + 1) % 10 == 0 or True:
#         elapsed = time.time() - start_time
#         avg_reward = np.mean(training_rewards[-10:])
#         print(f"Episode {episode+1:3d}/{NUM_EPISODES}  |  Loss: {loss:8.4f}  |  Avg Reward: {avg_reward:7.4f}  |  Time: {elapsed:.1f}s")


Starting initial compilation (this may take 2-5 minutes on a GTX 1070)...


/scratch/luisamao/conda/envs/cobot/lib/python3.10/site-packages/jax/_src/interpreters/xla.py:119: RuntimeWarning: overflow encountered in cast
  return np.asarray(x, dtypes.canonicalize_dtype(x.dtype))
2026-03-12 14:55:46.640949: E external/xla/xla/service/slow_operation_alarm.cc:73] 
********************************
[Compiling module jit_loss_fn] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
2026-03-12 14:55:50.172257: E external/xla/xla/service/slow_operation_alarm.cc:140] The operation took 2m3.531495212s

********************************
[Compiling module jit_loss_fn] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************


Compilation complete! Starting training loop...


## Section 6: Visualize Training Results

Plot training metrics and analyze policy performance.

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss plot
axes[0].plot(training_losses, linewidth=2)
axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Reward plot
axes[1].plot(training_rewards, linewidth=2, color='green', alpha=0.7)
axes[1].fill_between(range(len(training_rewards)), training_rewards, alpha=0.3, color='green')
axes[1].set_xlabel('Episode', fontsize=12)
axes[1].set_ylabel('Cumulative Reward', fontsize=12)
axes[1].set_title('Cumulative Reward Over Time', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Training Statistics:")
print(f"  - Initial reward: {training_rewards[0]:.4f}")
print(f"  - Final reward: {training_rewards[-1]:.4f}")
print(f"  - Best reward: {max(training_rewards):.4f}")
print(f"  - Reward improvement: {(training_rewards[-1] - training_rewards[0]):.4f}")

## Bonus: Policy Evaluation and Visualization

Test the trained policy on new trajectories and visualize the results.

In [ ]:
def evaluate_policy(params, num_rollouts=5):
    """Evaluate trained policy on new trajectories."""
    target_pos = jnp.array([0.3, 0.2, 0.3])
    
    all_rewards = []
    all_trajectories = []
    
    for rollout in range(num_rollouts):
        # Initialize environment
        key_init = jax.random.PRNGKey(1000 + rollout)
        data = reset_fn(key_init)
        obs = get_observation(data)
        
        trajectory = {'obs': [obs], 'actions': [], 'rewards': []}
        episode_reward = 0.0
        
        # Run episode
        for step in range(EPISODE_LENGTH):
            # Get action from policy
            action = policy.apply(params, obs)
            
            # Step environment
            data, obs, reward = step_fn(data, action, target_pos)
            
            trajectory['actions'].append(action)
            trajectory['obs'].append(obs)
            trajectory['rewards'].append(reward)
            episode_reward += reward
        
        all_rewards.append(episode_reward)
        all_trajectories.append(trajectory)
    
    return all_rewards, all_trajectories

# Evaluate the trained policy
print("Evaluating trained policy...")
eval_rewards, eval_trajectories = evaluate_policy(params, num_rollouts=5)

print(f"\nEvaluation Results ({len(eval_rewards)} rollouts):")
print(f"  - Mean reward: {np.mean(eval_rewards):.4f}")
print(f"  - Std reward:  {np.std(eval_rewards):.4f}")
print(f"  - Min reward:  {np.min(eval_rewards):.4f}")
print(f"  - Max reward:  {np.max(eval_rewards):.4f}")

# Plot evaluation rewards
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.bar(range(len(eval_rewards)), eval_rewards, color='blue', alpha=0.7)
plt.xlabel('Rollout', fontsize=12)
plt.ylabel('Cumulative Reward', fontsize=12)
plt.title('Policy Evaluation - Rollout Rewards', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

# Plot rewards per evaluation
plt.subplot(1, 2, 2)
step_rewards = np.array([[traj['rewards'][i] for i in range(len(traj['rewards']))] 
                          for traj in eval_trajectories])
plt.plot(step_rewards.T, alpha=0.6, linewidth=1.5)
plt.xlabel('Step', fontsize=12)
plt.ylabel('Reward', fontsize=12)
plt.title('Policy Evaluation - Step Rewards', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()